In [ ]:
#@title Estilo de la clase (ejecutar, no hace falta leer) {display-mode: "form"}
from IPython.display import HTML, display
display(HTML(r'''
<style>
@import url('https://fonts.googleapis.com/css2?family=Work+Sans:wght@400;600&family=Amiri:wght@400;700&display=swap');
.rendered_html, .markdown, .cell .text_cell_render { font-family:'Work Sans',system-ui,sans-serif; color:#122535; }
.rendered_html h1,.rendered_html h2,.rendered_html h3 { font-family:'Amiri',Georgia,serif; color:#00529B; }
.rendered_html h2 { border-bottom:2px solid #00529B; padding-bottom:.2em; }
.rendered_html a { color:#00529B; }
.rendered_html table th { background:#00529B; color:#fff; }
.rendered_html h1,.rendered_html h2,.rendered_html h3 { scroll-margin-top:16px; }
</style>
'''))

# Clase 3 · Visualización de datos

**Analítica de Datos** · Maestría en Ciencias del Comportamiento · Universidad de San Andrés

**Primavera 2026 · 22/08/2026**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomdamelio/analitica_de_datos_alumnos/blob/main/clases/clase-03/notebooks/clase03_python.ipynb)

---

Si se van a llevar una única idea de la clase de hoy, debe ser la siguiente:

> **Un gráfico es un mapeo de variables (datos) a atributos gráficos**, y de esto depende qué historia se puede contar.

**Cómo se usa esta notebook.** Las celdas de código se corren con `Shift+Enter`, de arriba hacia abajo. Si te salteás una, las de abajo pueden fallar porque dependen de variables definidas antes.

## La bibliografía de hoy

Toda la clase sale de un solo libro:

> Wilke, C. O. (2019). *Fundamentals of Data Visualization*. O'Reilly.
> Libro web completo, en inglés: **[clauswilke.com/dataviz](https://clauswilke.com/dataviz/)**

No hay que leerlo entero. Cada sección de esta notebook sale de un capítulo, y son cortos:

| Sección de hoy | Capítulos de Wilke |
|---|---|
| **1. El criterio** | [1. Introduction](https://clauswilke.com/dataviz/introduction.html) |
| **2. El mapeo** | [2. Mapping data onto aesthetics](https://clauswilke.com/dataviz/aesthetic-mapping.html) |
| **3. El repertorio** | [5. Directory of visualizations](https://clauswilke.com/dataviz/directory-of-visualizations.html), [6. Visualizing amounts](https://clauswilke.com/dataviz/visualizing-amounts.html), [7. Histograms and density plots](https://clauswilke.com/dataviz/histograms-density-plots.html), [9. Visualizing many distributions at once](https://clauswilke.com/dataviz/boxplots-violins.html) |
| **4. Las decisiones** | [3. Coordinate systems and axes](https://clauswilke.com/dataviz/coordinate-systems-axes.html), [4. Color scales](https://clauswilke.com/dataviz/color-basics.html), [19. Common pitfalls of color use](https://clauswilke.com/dataviz/color-pitfalls.html), [26. Don’t go 3D](https://clauswilke.com/dataviz/no-3d.html) |
| **5. Componer y comunicar** | [21. Multi-panel figures](https://clauswilke.com/dataviz/multi-panel-figures.html), [22. Titles, captions, and tables](https://clauswilke.com/dataviz/figure-titles-captions.html) |
| **6. El informe** | [29. Telling a story and making a point](https://clauswilke.com/dataviz/telling-a-story.html) |


Y hay dos referencias para consultar mientras hacen visualizaciones: la
[galería de seaborn](https://seaborn.pydata.org/examples/index.html) y
[from Data to Viz](https://www.data-to-viz.com/).


## La idea de la clase, en dos figuras

Antes de ponernos a trabajar, un ejemplo prestado del [capítulo 2](https://clauswilke.com/dataviz/aesthetic-mapping.html) de [Wilke](https://clauswilke.com/dataviz/).

Los datos son las **temperaturas diarias** de cuatro ciudades de Estados
Unidos: el promedio histórico de cada día del año, medido por el servicio meteorológico
estadounidense (NOAA). Cada fila es un día en una ciudad, y hay cuatro columnas: la
**ciudad**, el **día del año**, el **mes** y la **temperatura**.

Acá abajo están las primeras filas de esa tabla, y después la figura que se arma con
esas cuatro columnas. Mirala un minuto antes de seguir leyendo, porque la vamos a
desarmar entre todos.

In [ ]:
#@title Los datos: temperaturas diarias de la NOAA {display-mode: "form"}
# Descarga y prepara los datos originales del libro.
# No hace falta leer este codigo: lo que importa es la tabla que sale abajo.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs("figuras", exist_ok=True)

# Servicio publico de la NOAA: temperatura media normal (promedio 1981-2010) de cada
# dia del año, para las mismas cuatro estaciones que usa el libro.
URL_NOAA = ("https://www.ncei.noaa.gov/access/services/data/v1"
            "?dataset=normals-daily"
            "&stations=USW00014819,USC00042319,USW00093107,USW00012918"
            "&dataTypes=DLY-TAVG-NORMAL"
            "&startDate=2010-01-01&endDate=2010-12-31"
            "&format=csv&units=standard")
temperaturas = pd.read_csv(URL_NOAA)

nombres_ciudad = {"USW00014819": "Chicago",
                  "USC00042319": "Valle de la Muerte",
                  "USW00093107": "San Diego",
                  "USW00012918": "Houston"}
temperaturas["ciudad"] = temperaturas["STATION"].map(nombres_ciudad)

# La columna DATE trae "01-01", "01-02", ...: le pegamos un año bisiesto para poder
# leerla como fecha y sacar el dia del año y el mes.
fechas = pd.to_datetime("2012-" + temperaturas["DATE"])
temperaturas["dia_del_año"] = fechas.dt.dayofyear
temperaturas["mes"] = fechas.dt.month

# El libro las muestra en grados Fahrenheit; acá las pasamos a Celsius.
temperaturas["temperatura"] = (temperaturas["DLY-TAVG-NORMAL"] - 32) * 5 / 9

# Las cuatro columnas que importan hoy. Cada fila es un dia en una ciudad.
temperaturas[["ciudad", "dia_del_año", "mes", "temperatura"]].head()


In [ ]:
#@title Temperaturas normales diarias en cuatro ciudades (Wilke, cap. 2, fig. 2.3) {display-mode: "form"}
# Recreacion de la figura del libro con los datos de arriba.
# No hace falta leer este codigo: lo que importa es la figura.
orden_ciudades = ["Valle de la Muerte", "Houston", "San Diego", "Chicago"]

plt.figure(figsize=(9, 4))
sns.lineplot(data=temperaturas, x="dia_del_año", y="temperatura",
             hue="ciudad", hue_order=orden_ciudades, linewidth=1.6)
plt.xlabel("día del año")
plt.ylabel("temperatura (°C)")
plt.title("Temperaturas normales diarias en cuatro ciudades de EE.UU.")
plt.savefig("figuras/clase03_wilke_temperaturas_lineas.png", dpi=150, bbox_inches="tight")
plt.show()

*Recreación de la figura 2.3 de Wilke (cap. 2) con los datos originales. Fuente: NOAA.
El libro las muestra en grados Fahrenheit; acá están en Celsius.*

Esa figura es un mapeo, y se escribe entero en tres renglones de código. Para entender esto, elegí abajo en
cada desplegable el atributo gráfico que corresponda, y **volvé a correr la celda** para ver si acertaste.

In [ ]:
#@title ✏️ ¿Qué está mapeado a qué? {display-mode: "form"}
#@markdown **1. La temperatura, ¿a qué atributo gráfico está mapeada?**
temperatura = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **2. ¿Y el día del año?**
dia = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **3. ¿Y la ciudad?**
ciudad = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **4. La escala que lleva la ciudad, ¿de qué tipo es?**
escala_ciudad = "elegir" #@param ["elegir", "continua", "discreta"]

respuestas = [("1. temperatura", temperatura, "posición en el eje y"),
              ("2. día del año", dia, "posición en el eje x"),
              ("3. ciudad", ciudad, "color"),
              ("4. escala de la ciudad", escala_ciudad, "discreta")]

for titulo, elegida, correcta in respuestas:
    if elegida == "elegir":
        print("⚪", titulo, "— todavía sin responder")
    elif elegida == correcta:
        print("✅", titulo, "—", elegida)
    else:
        print("❌", titulo, "—", elegida, "  ·  la respuesta es:", correcta)

No hay una cuarta cosa escondida: la figura **es** esas tres decisiones.

In [ ]:
#@title Los mismos datos, otro mapeo (Wilke, cap. 2, fig. 2.4) {display-mode: "form"}
# Ahora la temperatura va al COLOR y la ciudad pasa a ser una posicion.
# Se promedia por mes para que cada cuadrado sea lo bastante grande como para
# que el color se pueda leer.
promedios = temperaturas.groupby(["ciudad", "mes"], as_index=False)["temperatura"].mean()
tabla_meses = promedios.pivot(index="ciudad", columns="mes", values="temperatura")

# De la mas fria a la mas calurosa, y los meses con nombre.
tabla_meses = tabla_meses.loc[["Chicago", "San Diego", "Houston", "Valle de la Muerte"]]
tabla_meses.columns = ["ene", "feb", "mar", "abr", "may", "jun",
                       "jul", "ago", "sep", "oct", "nov", "dic"]

plt.figure(figsize=(9, 2.6))
sns.heatmap(tabla_meses, cmap="inferno", cbar_kws={"label": "temperatura (°C)"})
plt.xlabel("mes")
plt.ylabel("")
plt.title("Temperatura normal media por mes y ciudad")
plt.savefig("figuras/clase03_wilke_temperaturas_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

*Recreación de la figura 2.4 de Wilke (cap. 2). Fuente: NOAA.*

Son **los mismos datos**. Lo único que cambió es qué variable fue a qué atributo, así que
el mapeo se puede cambiar. Las dos últimas preguntas tienen trampa.

In [ ]:
#@title ✏️ El mismo dato, otro mapeo {display-mode: "form"}
#@markdown **1. Acá, ¿a qué atributo gráfico está mapeada la temperatura?**
temperatura = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **2. ¿Y la ciudad?**
ciudad = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **3. El eje horizontal (los meses), ¿qué tipo de escala es?**
eje_horizontal = "elegir" #@param ["elegir", "continua", "discreta"]
#@markdown **4. ¿Y el eje vertical (las ciudades)?**
eje_vertical = "elegir" #@param ["elegir", "continua", "discreta"]

respuestas = [("1. temperatura", temperatura, "color"),
              ("2. ciudad", ciudad, "posición en el eje y"),
              ("3. eje horizontal", eje_horizontal, "discreta"),
              ("4. eje vertical", eje_vertical, "discreta")]

for titulo, elegida, correcta in respuestas:
    if elegida == "elegir":
        print("⚪", titulo, "— todavía sin responder")
    elif elegida == correcta:
        print("✅", titulo, "—", elegida)
    else:
        print("❌", titulo, "—", elegida, "  ·  la respuesta es:", correcta)

Las dos últimas son la trampa, y casi nadie las marca: en esta figura **los dos ejes son
escalas de posición discretas**. El eje x no es el tiempo continuo, son doce categorías
ordenadas; el eje y son cuatro ciudades, que ni siquiera tienen un orden natural. El orden
en que las pusimos es una decisión nuestra, no un dato: *"podría haber elegido cualquier
otro orden y la figura habría sido igual de válida"* (Wilke, cap. 2).

La idea con la que Wilke abre el capítulo 2 es la más importante de la clase:

> *"Todas las visualizaciones de datos mapean valores de los datos a características
> cuantificables del gráfico resultante. A esas características las llamamos **atributos
> gráficos** (aesthetics)."* (Wilke, cap. 2)

Todo gráfico, por distinto que parezca un grafico de barras de un gráficos de torta, o de un heatmap, es la
misma operación: **mapear valores de los datos a atributos gráficos**. El andamiaje
completo son cuatro piezas:

1. **Los atributos gráficos** (*aesthetics*, el término del libro): posición, forma, tamaño, color, ancho de
   línea, tipo de línea. La lista es corta y cerrada.
2. **Los tipos de dato**: continuo (entre dos valores siempre hay uno intermedio),
   discreto (categórico sin orden, categórico ordenado)
3. **La partición**: posición, tamaño, color y ancho de línea pueden representar datos
   continuos; forma y tipo de línea, en general, solo discretos.
4. **Las escalas**: el diccionario que traduce cada valor de dato a un valor del
   atributo. La única regla dura del capítulo: la escala tiene que ser **uno a uno**,
   o sea que a cada valor del dato le corresponde exactamente un valor del atributo, y
   al revés. Si dos sedes comparten el mismo color, el gráfico no es feo ni malo: es
   incorrecto, porque quedó ambiguo.

   Ojo que *uno a uno* no quiere decir *lineal*. Una escala logarítmica, una raíz
   cuadrada o un eje dado vuelta siguen siendo uno a uno: cada valor tiene su único
   lugar y desde el gráfico se puede volver al dato. Lo que se elige ahí es la **forma**
   de la traducción, y esa elección es libre siempre que quede declarada (el eje avisa
   que es logarítmico). Lo que el capítulo prohíbe es otra cosa: que dos valores
   distintos terminen en el mismo lugar.

El inventario de atributos gráficos, dibujado:

<img src="https://clauswilke.com/dataviz/aesthetic_mapping_files/figure-html/common-aesthetics-1.png" width="620">

*Figura 2.1 de Claus O. Wilke,
[Fundamentals of Data Visualization](https://clauswilke.com/dataviz/) (cap. 2),
reproducida con atribución bajo licencia CC BY-NC-ND 4.0.*

### Una más difícil

Con el inventario a la vista, una figura más cargada, también del libro. Son 32 autos de
1973-74, y usa **cinco escalas a la vez**.

Antes de verla, el punto de partida: los mismos autos con **dos** variables, la cilindrada
y la eficiencia del combustible. Dos escalas de posición y nada más. Este es el gráfico
más común que existe.

In [ ]:
#@title Los mismos 32 autos, con dos variables y nada más {display-mode: "form"}
# Los datos de la figura del libro (Motor Trend, 1974), desde su fuente publica.
# No hace falta leer este codigo: lo que importa es la figura.
autos = pd.read_csv("https://vincentarelbundock.github.io/Rdatasets/csv/datasets/mtcars.csv")

plt.figure(figsize=(6, 4))
sns.scatterplot(data=autos, x="disp", y="mpg", s=55)
plt.xlabel("cilindrada (displacement, pulgadas cúbicas)")
plt.ylabel("eficiencia del combustible (mpg)")
plt.title("32 autos de 1973-74: dos variables, dos escalas de posición")
plt.show()

Ahora la misma figura, con tres variables más encima. No cambia el tipo de
gráfico: siguen siendo los mismos 32 puntos en los mismos dos ejes.

Mirala y contestá, para cada variable: **¿a qué atributo gráfico está mapeada, y esa escala
es continua o discreta?**

<img src="https://clauswilke.com/dataviz/aesthetic_mapping_files/figure-html/mtcars-five-scale-1.png" width="560">

*Figura 2.5 de Claus O. Wilke,
[Fundamentals of Data Visualization](https://clauswilke.com/dataviz/) (cap. 2),
reproducida con atribución bajo licencia CC BY-NC-ND 4.0. Fuente de los datos:
Motor Trend, 1974.*

In [ ]:
#@title ✏️ Las cinco escalas de la figura 2.5 {display-mode: "form"}
#@markdown **1. La potencia (*power*), ¿a qué atributo está mapeada?**
potencia = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **2. ¿Y el peso (*weight*)?**
peso = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **3. ¿Y los cilindros (*cylinders*)?**
cilindros = "elegir" #@param ["elegir", "posición en el eje x", "posición en el eje y", "color", "tamaño", "forma"]
#@markdown **4. De las cinco variables, ¿cuál es la única que NO es continua?**
no_continua = "elegir" #@param ["elegir", "cilindrada (displacement)", "eficiencia del combustible", "potencia", "peso", "cilindros"]

respuestas = [("1. potencia", potencia, "color"),
              ("2. peso", peso, "tamaño"),
              ("3. cilindros", cilindros, "forma"),
              ("4. la única no continua", no_continua, "cilindros")]

for titulo, elegida, correcta in respuestas:
    if elegida == "elegir":
        print("⚪", titulo, "— todavía sin responder")
    elif elegida == correcta:
        print("✅", titulo, "—", elegida)
    else:
        print("❌", titulo, "—", elegida, "  ·  la respuesta es:", correcta)

Dos cosas para registrar antes de arrancar con lo nuestro.

La primera: **una variable más no pide necesariamente un gráfico nuevo, pide en principio un atributo más**. Arriba tenemos el
mismo diagrama de dispersión de recién, con tres canales extra encendidos. Y fijate cuál
se llevó la forma: los cilindros, la única variable discreta de las cinco. No es casualidad,
es la partición del punto 3 de acá arriba: la forma no sabe representar valores continuos.

La segunda: los canales **no rinden todos igual**. La cilindrada y la eficiencia se leen sin
esfuerzo; la potencia, más o menos; el peso y los cilindros, apenas. Cada canal que se
enciende cuesta legibilidad. Sobre el final de la clase vamos a ver qué pasa cuando se
encienden todos juntos.

Con esto ya tenemos la idea de la clase. Ahora, el trabajo.

Hoy, **ustedes son el equipo de ciencia de datos de Nimbus.**
Recursos Humanos quiere saber si conviene extender el piloto de fruta a todas las sedes
el año que viene. 

| # | Paso | Qué hacemos |
|---|---|---|
|0|[Preparación](#scrollTo=prep)| cargar los datos|
| 1 | [El criterio](#scrollTo=sec-criterio) | auditar los gráficos que ya mandó RRHH |
| 2 | [El mapeo](#scrollTo=sec-mapeo) | cómo se escribe un mapeo en seaborn |
| 3 | [El repertorio](#scrollTo=sec-repertorio) | qué gráfico contesta cada pregunta |
| 4 | [Las decisiones](#scrollTo=sec-decisiones) | dos gráficos correctos, ¿cuál mandamos? |
| 5 | [Componer y comunicar](#scrollTo=sec-componer) | varias figuras que hablen el mismo idioma |
| 6 | [El informe](#scrollTo=sec-informe) | la figura final y la conclusión |

Casi todos los pasos terminan con una **consigna corta** o con un quiz que se corrige
solo. Están pensadas para resolverse en cinco o diez minutos durante la clase.

## 0. Preparación

Hoy no hay que subir nada a Drive: **los datos se leen directo por URL** desde el sitio de
la materia. Son los tres archivos de Nimbus que ya conocés de la Clase 2.

La celda de abajo también crea una carpeta
`figuras/`: cada figura del informe la vamos a guardar ahí como archivo `.png` con
`plt.savefig(...)`. Eso es parte del tema de hoy: las figuras de un informe no viven en
la pantalla de colab, viven en archivos independientes que se pueden mandar, regenerar, etc.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Los datos se leen por URL: no hay que bajar ni montar nada.
# (parse_dates le avisa a pandas que "fecha" es una fecha, no un texto.)
BASE = "https://analiticadedatos-udesa.com/data/toy-nimbus/"
empleados = pd.read_csv(BASE + "nimbus_empleados.csv")
salario   = pd.read_csv(BASE + "nimbus_salario.csv")
bienestar = pd.read_csv(BASE + "nimbus_bienestar_diario.csv", parse_dates=["fecha"])

# Una carpeta para ir guardando las figuras del informe.
import os
os.makedirs("figuras", exist_ok=True)

print("empleados:", empleados.shape, "| salario:", salario.shape, "| bienestar:", bienestar.shape)

In [ ]:
#@title Estilo de las figuras del informe (ejecutar, no hace falta leer) {display-mode: "form"}
# Colores y estilo de la catedra: hace que todas las figuras del informe hablen el mismo
# idioma visual. Por que eso importa se ve en la seccion 5.
import seaborn as sns

PALETA = ["#00529B", "#C0492F", "#7E9EBB", "#1F7A4D", "#B4232E", "#9FB0BD"]
sns.set_theme(style="whitegrid", palette=PALETA)

### Como repaso de la clase pasada, ¿qué es una fila en cada tabla?

Los tres archivos hablan de la misma gente, pero **cada uno tiene su propia unidad de
observación**: lo que representa una fila cambia de tabla en tabla. Esa es la pregunta
que hay que contestar *antes* de hacer cualquier `merge`.

| tabla | una fila es... | filas |
|---|---|---|
| `empleados` | un empleado | 600 |
| `salario` | un empleado **en un año** | 1.800 = 600 × 3 años |
| `bienestar` | un empleado **en un día** | 24.000 = 600 × 40 días |

`merge` pega las columnas de una tabla en otra usando una columna en común, acá
`empleado_id`. Lo importante es anticipar qué le pasa a las filas:

```
   empleados (600 filas)                 bienestar (24.000 filas)
   1 fila = 1 empleado                   1 fila = 1 empleado en 1 día

   empleado_id  sede          area       empleado_id  fecha       bienestar
        1       Buenos Aires  Ventas          1       2025-03-03      4
        2       Mar del Plata Ingenieria      1       2025-03-04      5
        3       Mendoza       Producto        1       2025-03-05      3
       ...                                    1          ...        ...
                                              2       2025-03-03      6
        │                                     │
        └───────────  on="empleado_id"  ──────┘

   El empleado 1 aparece 1 vez a la izquierda y 40 veces a la derecha:
   "Buenos Aires" y "Ventas" van a quedar REPETIDOS en sus 40 filas diarias.
```

El resultado no tiene 600 filas ni 24.600: sigue teniendo **24.000**. El merge no agrega
filas, agrega **columnas**. Y eso es justo lo que necesitamos hoy: que cada medición
diaria sepa de qué sede y de qué área viene, para después poder pedirle a seaborn (libreria de Python de visualizacion de datos)
"separame por sede" o "coloreame por área".

In [ ]:
# Miremos las tres tablas antes de pegarlas.
print("empleados:", empleados.shape)
print(empleados.head(3))

print()
print("salario:", salario.shape)
print(salario.head(3))

print()
print("bienestar:", bienestar.shape)
print(bienestar.head(3))

# La pregunta que decide como sale el merge: ¿cuantas veces aparece UN empleado?
print()
print("el empleado 1 aparece...")
print("  en empleados:", len(empleados[empleados["empleado_id"] == 1]), "vez")
print("  en salario:  ", len(salario[salario["empleado_id"] == 1]), "veces (una por año)")
print("  en bienestar:", len(bienestar[bienestar["empleado_id"] == 1]), "veces (una por dia)")

In [ ]:
# MERGE 1: a cada empleado le pegamos su salario.

# Ojo: salario tiene 3 filas por empleado (2023, 2024, 2025). Si las pegaramos
# todas, cada empleado aparecería 3 veces en el resultado. Como para el informe
# alcanza con el sueldo actual, primero nos quedamos con un solo año.
salario_2025 = salario[salario["anio"] == 2025]
print("salario:", salario.shape, "  ->  salario_2025:", salario_2025.shape)

# Ahora si: 1 fila por empleado de cada lado -> 1 fila por empleado en el resultado.
emp = empleados.merge(salario_2025, on="empleado_id")
print("emp:", emp.shape, "  (las mismas 600 personas, con 3 columnas nuevas)")

emp.head(3)

El segundo merge es de otro tipo, y es el que conviene mirar con atención.

Acá el de la izquierda es `bienestar`, que tiene **40 filas por empleado**. Cada
empleado aparece una sola vez en `empleados`, así que su sede y su área se van a
**copiar en las 40 filas** que le corresponden. La tabla no se agranda a lo largo
(sigue teniendo 24.000 filas), se agranda a lo ancho.

In [ ]:
# MERGE 2: a cada medicion diaria le pegamos la sede y el area de quien la reporto.

# De empleados nos llevamos SOLO las columnas que vamos a usar, mas la clave.
# (Si pegaramos la tabla entera, arrastrariamos genero, educacion, etc. sin necesidad.)
datos_del_empleado = empleados[["empleado_id", "sede", "area"]]

bien = bienestar.merge(datos_del_empleado, on="empleado_id")
print("bien:", bien.shape, "  (las mismas 24.000 mediciones, con 2 columnas nuevas)")

# La prueba de lo que dijimos arriba: las primeras filas del empleado 1.
# "Buenos Aires" y "Ventas" aparecen repetidos, uno por cada dia que reporto.
bien[bien["empleado_id"] == 1].head(4)

Tres preparativos más y ya podemos graficar.

El primero es **una columna que no existía**: la fecha nos dice el día exacto, pero
para el informe vamos a querer hablar de *semanas del piloto*. La construimos.

El segundo es **una decisión de informe**, y conviene tomarla ahora y no cuando
aparezca el primer gráfico: fijar el orden de los grupos (Tratamiento y Control). Si no lo hacemos, cada
gráfico los ordena como quier (i.e. primero Tratamiento y despues Control, o al reves), y el informe termina estando desordenado (algo
que vamos a criticar más adelante en los gráficos de otros).

El tercero es **una unidad**. Los salarios de Nimbus tienen siete dígitos, y un eje
lleno de números como 1250000 no lo lee nadie. Nos guardamos el salario también en
millones de pesos, y de ahí en adelante graficamos esa columna, diciendo la unidad en
el título del eje.

In [ ]:
# La fecha dice el dia exacto (2025-03-03). Para el informe queremos algo mas simple:
# en que SEMANA del piloto estamos. Se construye en dos pasos.

primer_dia = bien["fecha"].min()
print("el piloto arranco el", primer_dia.date())

# PASO 1: cuantos dias pasaron desde ese primer dia.
#   Restar dos fechas no da un numero, da una duracion (pandas la muestra como
#   "7 days"). El accesorio .dt.days se queda con el numero pelado.
bien["dias_desde_el_inicio"] = (bien["fecha"] - primer_dia).dt.days

# PASO 2: cada bloque de 7 dias es una semana.
#   // es la division ENTERA: divide y tira los decimales.
#   dias 0 a 6  -> 0     dias 7 a 13 -> 1     dias 14 a 20 -> 2
#   El +1 es para que la primera semana se llame 1 y no 0.
bien["semana"] = bien["dias_desde_el_inicio"] // 7 + 1

# Miremoslo funcionando, en los primeros dias de un empleado:
print()
print(bien[bien["empleado_id"] == 1][["fecha", "dias_desde_el_inicio", "semana"]].head(11).to_string(index=False))

Mirá el salto del día 4 al día 7: los días 5 y 6 son sábado y domingo, y el piloto
sólo tiene días hábiles, así que **no existen en los datos**. La cuenta funciona igual
porque contamos *días de calendario* desde el arranque, no filas de la tabla. Si
hubiéramos contado filas ("cada 5 mediciones, una semana"), el fin de semana la habría
roto.

Ahora sí, con la semana ya construida, podemos ver la forma del piloto:

In [ ]:
# Cada fase cubre 4 semanas: primero todos sin fruta, despues solo un grupo con fruta.
print(bien.groupby("fase")["semana"].unique())

In [ ]:
# 2) El orden de los grupos, fijado de una vez para toda la clase:
#    Control primero (navy), Tratamiento segundo (terracota).
#    pd.Categorical le pega ese orden a la columna, y seaborn lo respeta al
#    asignar colores y al ubicar las categorias en los ejes.
ORDEN_GRUPO = ["Control", "Tratamiento"]
bien["grupo_fruta"] = pd.Categorical(bien["grupo_fruta"], ORDEN_GRUPO, ordered=True)
emp["grupo_fruta"] = pd.Categorical(emp["grupo_fruta"], ORDEN_GRUPO, ordered=True)

print("orden fijado:", list(bien["grupo_fruta"].cat.categories))

# 3) El salario en millones, para que los ejes se lean.
emp["salario_millones"] = emp["salario_mensual"] / 1_000_000

print(emp[["salario_mensual", "salario_millones"]].head(3).to_string(index=False))

Y con esto ya tenemos el número que RRHH quiere entender.

In [ ]:
# groupby con DOS columnas: devuelve una fila por cada combinacion de fase y grupo.
print(bien.groupby(["fase", "grupo_fruta"])["bienestar"].mean().round(2))

(*Paréntesis*: El FutureWarning que sale arriba es un aviso de la librería pandas de que en futuras versiones algo que estás haciendo quizás funcione distinto o directamente no funcione. Sirve para mantener el código actualizado, y que no se rompa al cambiar de versiones.)

Ahí está el panorama. En la fase *baseline*, antes de que hubiera fruta en ningún
lado, los dos grupos están prácticamente iguales: 4,62 y 4,60. En la fase de
intervención, el grupo con fruta marca 4,95 y el de control se queda en 4,61.

Ahora, cuidado con lo que podemos afirmar. **Son cuatro promedios.** Un promedio solo
no distingue una diferencia real del ruido: no dice nada sobre cuánto varían las
respuestas, ni sobre cuánta gente hay detrás de cada número. Decidir si una diferencia
como esta es atribuible al programa, y no al azar, es una pregunta de **inferencia**,
con las herramientas que traen de Estadística y que acordamos como base en la Clase 1.

Y esa no es la pregunta de hoy. **La pregunta de hoy es cómo se muestra esto.** Un
gráfico honesto de estos datos tiene que dejar ver no sólo el tercio de punto de
diferencia, sino también cuánto se solapan los grupos, para que quien lo lea pueda
juzgar por su cuenta. Ahí está lo interesante del encargo: con estos mismos cuatro
números vamos a poder construir una figura que sugiera un efecto contundente y otra
que sugiera que no pasó nada, **sin cambiar un solo dato**.

## 1. El criterio

<a id="sec-criterio"></a>

Para analizar como se componen los gráficos, necesitamos un vocabulario, y lo tomamos del libro de la clase (Wilke, *Fundamentals of
Data Visualization*, cap. 1). Una figura puede fallar de tres maneras distintas:

| veredicto | el problema es de... | en palabras de Wilke |
|---|---|---|
| **fea** | estética | tiene problemas estéticos, pero por lo demás es clara e informativa |
| **mala** | percepción | es poco clara, confusa, innecesariamente complicada o engañosa |
| **incorrecta** | matemática | es objetivamente incorrecta |

Tres precisiones antes de usarlo:

- **No hay categoría "buena"**: toda figura sin veredicto es, al menos, aceptable.
- **Los bordes son difusos**, sobre todo entre fea y mala, y Wilke lo admite. Se puede
  discutir.
- **Es una escala de gravedad.** Pregunta para pensar: ¿qué preferirías que se mande,
  una figura fea o una mala?

Veámoslo primero con el ejemplo del propio libro recreado acá con tres números
inventados (A=3, B=5, C=4). Cuatro versiones del mismo dato, una por celda.

Las cuatro figuras de abajo son las del libro, recreadas. **NO hace falta que entiendas el código**: usan detalles de matplotlib que no vamos a enseñar hoy, y acá lo que
importa es mirarlas, no fabricarlas. Corré cada celda y mirá el resultado.

In [ ]:
#@title Figura 1.1 del libro, version (a): aceptable {display-mode: "form"}
# Los datos del ejemplo del libro: tres categorias, tres valores.
df_abc = pd.DataFrame({"tipo": ["A", "B", "C"], "valor": [3, 5, 4]})

# Version (a), la aceptable: un color, un eje que arranca en cero, nada de sobra.
plt.figure(figsize=(5, 3))                 # figsize es (ancho, alto) de la hoja
plt.bar(df_abc["tipo"], df_abc["valor"], color="#00529B")
plt.ylabel("valor")
plt.title("(a) aceptable")

# plt.savefig guarda la figura actual como archivo png; dpi es la resolucion.
plt.savefig("figuras/clase03_triada_a.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#@title Figura 1.1, version (b): FEA {display-mode: "form"}
# Version (b), la FEA: colores chillones que no codifican nada, grilla que grita,
# tipografias mezcladas. Los numeros se leen igual... pero cuesta mirarla.
plt.figure(figsize=(5, 3))
plt.bar(df_abc["tipo"], df_abc["valor"], color=["#aaff00", "#00ffff", "#cc00cc"])
plt.grid(color="black", linewidth=1.2)
plt.ylabel("valor", fontsize=15, family="serif")
plt.title("(b) FEA: correcta, pero cuesta mirarla", family="monospace")
plt.savefig("figuras/clase03_triada_b.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#@title Figura 1.1, version (c): MALA {display-mode: "form"}
# Version (c), la MALA: cada barra en su propio panel, y cada panel con SU eje.
# Ningun numero es falso, y sin embargo las tres barras "quedan iguales".
fig, axs = plt.subplots(1, 3, figsize=(8, 3))   # una hoja con tres paneles

axs[0].bar(["A"], [3], color="#00529B")
axs[0].set_ylim(0, 3.2)          # el eje de A llega hasta 3.2

axs[1].bar(["B"], [5], color="#00529B")
axs[1].set_ylim(0, 5.2)          # el de B hasta 5.2

axs[2].bar(["C"], [4], color="#00529B")
axs[2].set_ylim(0, 4.3)          # el de C hasta 4.3

fig.suptitle("(c) MALA: tres ejes distintos, las barras 'quedan iguales'")
plt.savefig("figuras/clase03_triada_c.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#@title Figura 1.1, version (d): INCORRECTA {display-mode: "form"}
# Version (d), la INCORRECTA: sin escala en el eje y, los numeros no se pueden
# recuperar. La figura dejo de ser una afirmacion verificable.
plt.figure(figsize=(5, 3))
plt.bar(df_abc["tipo"], df_abc["valor"], color="#00529B")
plt.yticks([])                   # lista vacia = borrar las marcas del eje y
plt.title("(d) INCORRECTA: no hay forma de leer los valores")
plt.savefig("figuras/clase03_triada_d.png", dpi=150, bbox_inches="tight")
plt.show()

La versión (c) es la más peligrosa de las cuatro: ningún número está mal, ningún eje está
truncado, cada panel leído por separado es honesto. Y sin embargo la comparación visual
miente, porque B ya no se ve mayor que A. Guardá esa idea: vuelve al final de la clase.

Ahora sí, los dos gráficos que mandó RRHH, hechos con los datos reales de Nimbus.

### Un minuto de matplotlib, y no volvemos

**`seaborn` no dibuja solo**: se apoya en otra librería, `matplotlib`, que es la que pone la
hoja en blanco y la muestra en pantalla. Por eso, cada tanto, van a ver líneas que empiezan
con `plt.` en vez de `sns.`. No vamos a aprender matplotlib, pero sí conviene tener claras
**dos palabras**, porque aparecen todo el tiempo:

- **la figura** es la **hoja entera**: lo que termina guardado en el archivo `.png`. Puede
  tener un panel o varios.
- **los ejes** indican **los paneles de esa hoja**: el **panel** es el rectángulo donde efectivamente se dibuja,
  con su eje x, su eje y y su título. El nombre es desafortunado: "los ejes" no son las dos
  rayas, son las coordenadas de un panel completo.

**El caso simple, que es el que más van a ver.** Si la figura tiene un solo panel, no hay
que pedir nada: se llama a seaborn y matplotlib crea la hoja sola, por atrás.

```
sns.countplot(data=empleados, y="sede", color="#00529B")
plt.title("Empleados por sede")     # el titulo de la figura
plt.xlabel("empleados")             # el titulo del eje x
plt.ylabel("")                      # el del eje y; vacio = sin titulo
plt.savefig("figuras/sedes.png")    # guarda la hoja como archivo
plt.show()                          # la muestra
```

**El caso con varios paneles.** Ahí sí hay que pedir la hoja de antemano, y decirle cuántos
paneles tiene:

```
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
```

Esa línea se lee así:

- `plt.subplots(1, 2)` pide una hoja de **1 fila y 2 columnas** de paneles. Los dos números
  son eso: primero filas, después columnas. `plt.subplots(2, 3)` daría seis paneles.
- `figsize=(11, 4)` es el **tamaño de la hoja** en pulgadas: primero el ancho, después el
  alto. Es opcional.
- A la izquierda del `=` hay **dos nombres** porque la función devuelve dos cosas de una
  vez: `fig` es la hoja entera y `axs` son sus paneles. Los nombres son convención, podrían
  ser cualquier otro.

Con más de un panel, `axs` no es un panel sino **la lista de paneles**, y a cada uno se
llega por su número empezando en cero: `axs[0]` es el de la izquierda, `axs[1]` el de la
derecha. (De ahí el plural.) Y como ahora hay dónde elegir, hay que decirle a seaborn **en
cuál de los dos dibujar**, con `ax=`:

```
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=bien, x="fase", y="bienestar", ax=axs[0])       # panel izquierdo
axs[0].set_title("Boxplot")
sns.violinplot(data=bien, x="fase", y="bienestar", ax=axs[1])    # panel derecho
axs[1].set_title("Violin")
plt.show()
```

Última diferencia, y con esto ya está: cuando hay varios paneles, el título **es de cada
panel**, así que se pide sobre él (`axs[0].set_title(...)`) y no con `plt.title(...)`, que
no sabría a cuál de los dos ponérselo. Lo mismo con `set_xlabel` y `set_ylabel`.

Eso es todo lo prestado por matplotlib. La división del trabajo, en una línea: **matplotlib pone la hoja,
seaborn dibuja encima**.

# Ahora si, volvemos a Nimbus

RRHH no nos esperó: ya armó sus propios gráficos en una planilla y los adjuntó al pedido.
Antes de producir nada, nuestro primer trabajo es **auditar** lo que mandaron.

Antes de mirarlos, un minuto de sintaxis, porque de acá en adelante todos los gráficos
de la clase se escriben igual. La librería que usamos es **seaborn**, y una llamada
típica se lee así:

```
sns.countplot( data=empleados ,  y="area" ,  color="#00529B" )
     ↑              ↑               ↑              ↑
  qué gráfico    de qué tabla    qué columna    de qué color
  quiero        salen los       va al eje y    lo pinto
                datos
```

Tres cosas que conviene registrar desde ahora:

- **`sns.`** adelante: es la librería. Todas las funciones que empiezan así dibujan
  algo.
- **`data=`** siempre recibe una tabla entera, y las demás opciones reciben **nombres
  de columnas de esa tabla**, escritos entre comillas. Nunca se le pasan los datos
  sueltos.
- El nombre de la función dice **qué tipo de gráfico** es: `countplot` cuenta filas por
  categoría, `boxplot` hace cajas, `scatterplot` hace nubes de puntos.

El *por qué* la librería está diseñada así ya lo vimos al abrir la clase, y en la
sección 2 lo retomamos, ahora escrito en código. Por ahora alcanza con poder leer
las líneas.

In [ ]:
#@title El grafico que mando RRHH {display-mode: "form"}
# Ilustracion: las barras 3D necesitan matplotlib crudo, que no enseñamos hoy.
conteo_sedes = empleados["sede"].value_counts()
posiciones = list(range(len(conteo_sedes)))
ceros = [0] * len(conteo_sedes)
colores = plt.cm.rainbow([p / (len(posiciones) - 1) for p in posiciones])

fig = plt.figure(figsize=(8.5, 5))
ax = fig.add_subplot(projection="3d")
ax.bar3d(posiciones, ceros, ceros, 0.55, 0.55, conteo_sedes.values,
         color=colores, shade=True)
ax.set_xticks([p + 0.28 for p in posiciones])
ax.set_xticklabels(conteo_sedes.index, rotation=90, fontsize=8)
ax.set_yticks([])
ax.set_zlabel("cant", fontsize=9)
ax.tick_params(axis="z", labelsize=8)
ax.set_title("Empleados por sede", fontsize=14, family="serif")

# (sin bbox_inches="tight": con ejes 3D recorta mal el rotulo del eje z)
fig.subplots_adjust(right=0.82)
plt.savefig("figuras/clase03_auditoria_rrhh1.png", dpi=150)
plt.show()

Eso es lo que RRHH mandó adjunto al pedido: cuántos empleados tiene cada sede.

Antes de seguir, contestá dos preguntas mirando **sólo esta figura**.

1. **¿Qué sede tiene más empleados?** Anotá tu respuesta, la vamos a usar en un minuto.
2. **¿Qué le criticarías a este gráfico?** Hacé la lista de todo lo que te moleste, sin
   filtrar: los colores, las letras, las líneas, lo que sea.

Ahora hagamos nosotros el mismo gráfico, con los mismos datos y sin ningún adorno.

In [ ]:
# El mismo dato, en un grafico plano y de un solo color.
orden_sedes = empleados["sede"].value_counts().index   # las sedes, de mayor a menor

sns.countplot(data=empleados, y="sede", order=orden_sedes, color="#00529B")
plt.xlabel("empleados")
plt.ylabel("")
plt.title("Empleados por sede")

plt.savefig("figuras/clase03_auditoria_rrhh1_ok.png", dpi=150, bbox_inches="tight")
plt.show()

Empecemos por la pregunta que te pedí que te guardaras. Si mirando el gráfico de RRHH
dijiste **Mendoza**, es la respuesta que la figura invita a dar: es la barra de adelante y
se ve enorme. Pero Mendoza es la **más chica** de las seis (93 empleados). La más grande
es **Buenos Aires** (109), que allá atrás se ve modesta. La perspectiva sola podría dar vuelta el
orden real.

Ahora sí, la lista completa. Son cuatro defectos, y no son todos de la misma gravedad:

| defecto | veredicto | por qué |
|---|---|---|
| Un color por barra | **feo** | La sede ya está escrita abajo de cada barra: el color repite lo que el eje dice, y a cambio mete seis tonos que compiten entre sí |
| La grilla marcada | **feo** | Las líneas pesan casi tanto como los datos, cuando su único trabajo es ayudar a estimar de reojo |
| Los nombres verticales | **feo** | Hay que inclinar la cabeza para leer cada uno. Es lo primero que se le ocurre a cualquiera cuando las etiquetas no entran, y la salida no es rotarlas: es dar vuelta el gráfico |
| El 3D | **malo** | Cambia lo que percibís, no sólo lo que te cuesta leer |

Los tres primeros son molestos, pero no te impiden llegar al número correcto: te hacen
trabajar de más. Eso es **feo**.

El 3D es otra cosa, y por eso el veredicto cambia. La profundidad ahí **no codifica nada**:
las seis sedes están apoyadas en la misma línea, y la tercera dimensión es pura
decoración. Pero al proyectar un objeto 3D sobre una pantalla plana, lo que está más cerca
se agranda — así que la decoración **cambió lo que viste**. Sumale que el eje de la altura
quedó flotando en el medio, sin alinearse con ninguna barra, y que dice "cant" en vez de
decir qué se está contando.

Fijate que el gráfico de abajo no es más elaborado que el de arriba: es **menos**. Le
sacamos cosas, y por eso se lee.

Una última cosa sobre el 3D, porque es el caso más discutido de los cuatro: la intuición
dice *feo*, y Wilke lo etiqueta **malo**, pero con un argumento matemático (*incorrecto*). Es el mejor
ejemplo de que el borde entre malo e incorrecto es poroso.
Las excepciones para 3D que el libro sí
admite son las visualizaciones interactivas que se pueden rotar, u objetos que son
tridimensionales de verdad.

In [ ]:
#@title El segundo grafico que mando RRHH {display-mode: "form"}
# Salario promedio por sede, "para mostrar las brechas". Tal como llego.
# as_index=False deja "sede" como columna comun (no como indice): lo que seaborn espera.
medias_rrhh = emp.groupby("sede", as_index=False)["salario_mensual"].mean()
orden_rrhh = medias_rrhh.sort_values("salario_mensual", ascending=False)["sede"]

sns.barplot(data=medias_rrhh, x="salario_mensual", y="sede", order=orden_rrhh,
            color="#00529B")
plt.xlim(1_243_000, 1_272_000)
plt.ylabel("")
plt.title("La brecha salarial entre sedes")
plt.savefig("figuras/clase03_auditoria_rrhh2.png", dpi=150, bbox_inches="tight")
plt.show()

Ese es el segundo adjunto. RRHH lo mandó para mostrar las brechas entre sedes, y la
conclusión que sacaron es que **Córdoba paga muchísimo más que Mar del Plata**.

Otra vez, dos preguntas mirando sólo la figura.

1. **¿Cuánto más paga Córdoba que Mar del Plata?** Contestá con un número: ¿un 10% más?
   ¿el doble? ¿diez veces más?
2. **¿En qué número arranca el eje de abajo?** Miralo bien antes de seguir.

Ahora el mismo dato, en un gráfico igual en todo salvo en una cosa.

In [ ]:
# El mismo dato y el mismo tipo de grafico, con dos arreglos: el eje arranca en cero
# (una barra lo hace sola, si uno no le pide otra cosa) y el salario va en millones,
# con la unidad dicha en el titulo del eje.
medias_sede = emp.groupby("sede", as_index=False)["salario_millones"].mean()
orden_sedes = medias_sede.sort_values("salario_millones", ascending=False)["sede"]

sns.barplot(data=medias_sede, x="salario_millones", y="sede", order=orden_sedes,
            color="#00529B")
plt.xlabel("salario mensual promedio (millones de $)")
plt.ylabel("")
plt.title("Salario mensual promedio por sede")
plt.savefig("figuras/clase03_auditoria_rrhh2_ok.png", dpi=150, bbox_inches="tight")
plt.show()

**Veredicto: incorrecta.** Y acá no hay discusión de gusto posible.

Si contestaste "el doble" o "diez veces más", la figura hizo exactamente lo que sus autores
querían. La diferencia real entre Córdoba y Mar del Plata es del **2%**: 1.270.296 pesos
contra 1.245.407. Es lo que muestra el segundo gráfico, donde las seis barras se ven casi
iguales, porque casi iguales es lo que son.

El problema es de aritmética. Una barra dice "cuánto" con su **longitud**: la de Córdoba
tiene que ser un 2% más larga que la de Mar del Plata, porque el salario es un 2% mayor.
Al arrancar el eje en 1.243.000 en vez de en cero, cada barra dejó de medir el salario y
pasó a medir *cuánto sobra por encima de 1.243.000*, que no es un número que le interese a
nadie. Ahí la barra de Córdoba quedó más de diez veces más larga que la de Mar del Plata.

Y hay dos arreglos más que hicimos de paso, los dos de la categoría **fea**. El eje de
RRHH se titula `salario_mensual`, que es el nombre de la columna en la tabla y no dice
qué se está midiendo ni en qué unidad. Y arriba a la izquierda le quedó colgado un
`1e6`: es la forma que tiene matplotlib de avisar que hay que multiplicar cada número
por un millón. No es falso, pero obliga a hacer una cuenta mental para leer un salario,
y nadie en RRHH la va a hacer. Nuestra versión pasa los salarios a millones y lo aclara
en el título del eje.

Y ojo con la moraleja fácil: el problema no es cortar un eje, es cortarlo **con barras**.
Cuando lleguemos al repertorio de gráficos vamos a ver la salida legítima para mostrar una
diferencia chica sin exagerarla.

Con esto ya tenemos el criterio del día. Cada gráfico que aparezca de acá en adelante,
propio o ajeno, se puede someter a la misma pregunta: **¿es feo, malo o incorrecto?** (o
ninguna de las tres, que es lo que vamos a intentar).

Como dice Wilke al cerrar su primer capítulo: *"los animo a desarrollar su propio ojo y
a evaluar críticamente mis decisiones"*. Vale también para los gráficos de esta clase.

## 2. El mapeo, ahora en código

<a id="sec-mapeo"></a>

Al abrir la clase vimos la idea: un gráfico es un mapeo de variables a atributos
gráficos, y elegir el mapeo es elegir qué se puede leer. Lo que sigue es cómo se escribe
eso en seaborn, sobre nuestros propios datos.

### La gramática de seaborn es, literalmente, el mapeo

En seaborn cada argumento **es** una decisión de mapeo. Se le pasa la tabla en `data=` y
después se dice qué columna va a qué atributo:

| atributo (Wilke) | argumento (seaborn) |
|---|---|
| posición | `x=`, `y=` |
| color | `hue=` |
| forma | `style=` |
| tamaño | `size=` |
| ancho de línea | `size=` (en `lineplot`) |
| tipo de línea | `style=` (los patrones, con `dashes=`) |

Y con eso ya podemos hacer el movimiento central del bloque: **el mismo dato, dos veces,
con dos mapeos distintos**. Es el bienestar promedio por sede a lo largo del piloto, con las
mismas tres escalas (dos de posición y una de color) repartidas de dos maneras distintas.

In [ ]:
# Version 1 del mapeo: la semana al eje x, el bienestar al eje y, la sede al color.
# estimator="mean": si hay muchos valores por punto, dibuja el promedio.
sns.lineplot(data=bien, x="semana", y="bienestar", hue="sede",
             estimator="mean", errorbar=None)
plt.xlabel("semana del piloto")
plt.ylabel("bienestar promedio (1-7)")
plt.title("Mapeo 1: el tiempo en x, el bienestar en y, la sede en color")
plt.savefig("figuras/clase03_mapeo_lineas.png", dpi=150, bbox_inches="tight")
plt.show()

Esa es una de las dos. Veamos la otra, que la escribimos juntos acá en clase.

### El mismo dato, el otro mapeo

Queremos la **versión permutada**: los mismos números y las mismas tres escalas, pero con
el **bienestar en el color** y la **sede en el eje y**. Como el color necesita superficie
para poder leerse, cada combinación de sede y semana va a ser un cuadrado pintado.

Son dos decisiones, y conviene mirarlas en el código antes de correr la celda:

1. **La grilla.** `pivot_table` la arma: hay que decirle qué variable va a las *filas*
   (`index="sede"`), cuál a las *columnas* (`columns="semana"`) y cuál es el *valor* que
   se promedia dentro de cada celda (`values="bienestar"`).
2. **La función.** La de seaborn que pinta una grilla de cuadrados coloreados es
   `heatmap`.

In [ ]:
# pivot_table arma la grilla: una fila por sede, una columna por semana.
# aggfunc dice como resumir los valores que caen dentro de cada celda.
tabla = bien.pivot_table(index="sede", columns="semana", values="bienestar",
                         aggfunc="mean")

# annot=True escribe el numero en cada celda; fmt=".2f" son dos decimales.
sns.heatmap(tabla, cmap="viridis", annot=True, fmt=".2f",
            cbar_kws={"label": "bienestar promedio"})
plt.xlabel("semana del piloto")
plt.ylabel("")
plt.title("Mapeo 2: la sede en y, el bienestar en color")
plt.savefig("figuras/clase03_mapeo_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

Las dos figuras dicen **los mismos números**, y sin embargo no se leen igual. Probá
contestar con cada una estas dos preguntas: *¿qué sede estuvo más abajo durante toda la
semana 6?* y *¿qué sede cruzó a cuál entre la semana 3 y la 4?* Cada figura hace fácil una
de las dos, y trabajosa la otra.

¿Alguna está mal? **Ninguna**, igual que con las temperaturas del principio de la clase.

Y hay una decisión que `pivot_table` tomó por vos sin avisar: las sedes quedaron en orden
**alfabético**, porque es lo que hace por defecto. Cualquier otro orden daría una figura
distinta e igual de válida, y uno que dificulte la lectura sería *feo*, no un error.

## 3. El repertorio

<a id="sec-repertorio"></a>

Wilke organiza su catálogo de gráficos **por el mensaje que comunican**, no por el tipo
de dato: cantidades, distribuciones, proporciones, relaciones x-y, datos geoespaciales,
incertidumbre. La pregunta previa a todo gráfico del informe es entonces: **¿qué quiero
que se comunique?**

De las seis ramas hoy trabajamos dos a fondo, **cantidades** y **distribuciones**, que
son las que más van a usar; relaciones y proporciones aparecen en el juego y en las
consignas, y mapas e incertidumbre avanzada quedan sólo nombradas.

Un buen mapa
interactivo de todo el repertorio de gráficos es: [from Data to Viz](https://www.data-to-viz.com/)
(introduce temas por la forma de los datos; Wilke lo hace por el objetivo de comunicación).

### Cantidades: barras, y las decisiones que las rodean

La orientación ya la vimos en la auditoría. Falta el **orden de las barras**, que es una
decisión más consecuente de lo que parece.

La regla: *Si las barras representan categorías sin orden propio, ordenalas por valor*. El orden con el que vienen los datos (alfabético, o el de carga en la planilla) no significa nada, y dejarlo es una decisión que uno no tomó. Wilke marca ese
caso como **malo**, no como feo: no es que quede desprolijo, es que la comparación entre
barras se vuelve innecesariamente difícil.

Pero la regla tiene una excepción que casi siempre se olvida, y vale la pena verla:

In [ ]:
orden_natural = ["Secundario", "Terciario", "Universitario", "Posgrado"]
orden_por_valor = empleados["educacion_nivel"].value_counts().index

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
sns.countplot(data=empleados, x="educacion_nivel", order=orden_por_valor,
              color="#00529B", ax=axs[0])
axs[0].set_title("Ordenado por valor: prolijo... y confuso")
sns.countplot(data=empleados, x="educacion_nivel", order=orden_natural,
              color="#00529B", ax=axs[1])
axs[1].set_title("Orden natural: la forma de la distribucion aparece")
plt.tight_layout()
plt.show()

La regla "ordená las barras por valor" tiene una excepción que casi siempre se olvida:
vale **sólo para categorías sin orden propio**. Educación tiene un orden natural
(Secundario, Terciario, Universitario, Posgrado), y respetarlo hace aparecer la forma de
la distribución. Ordenarla por valor produce una figura más prolija y menos informativa:
técnicamente correcta, pero **mala**, porque destruye la lectura.

### Distribuciones

Un histograma no muestra "los datos": muestra una **interpretación** gobernada por el
ancho de *bin* (el ancho de la barra). Y si no lo elegiste vos, lo eligió el software.

Abajo está el mismo salario tres veces. El primer panel es lo que sale sin pedir nada:
**¿quién eligió ese ancho?** Un algoritmo, que no conoce ni los datos ni la pregunta.
Wilke lo dice sin vueltas: lo más probable es que ese ancho no sea el más apropiado para
el histograma que uno quiere hacer. De ahí la regla, que es corta: **probá varios anchos
antes de quedarte con uno.**

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 3.4))

sns.histplot(data=emp, x="salario_millones", ax=axs[0])
axs[0].set_title("Sin pedir nada: el ancho lo eligio seaborn")
axs[0].set_xlabel("salario mensual (millones de $)")

sns.histplot(data=emp, x="salario_millones", binwidth=0.01, ax=axs[1])
axs[1].set_title("binwidth=0,01 (10 mil $): ruido")
axs[1].set_xlabel("salario mensual (millones de $)")

sns.histplot(data=emp, x="salario_millones", binwidth=0.40, ax=axs[2])
axs[2].set_title("binwidth=0,40 (400 mil $): una sola barra")
axs[2].set_xlabel("salario mensual (millones de $)")

plt.tight_layout()
plt.show()
# Proba: cambia algun binwidth y volve a correr la celda.

### Boxplot y violín: ver la distribución en poco espacio

Cuando lo que importa es el **corrimiento entre grupos**, la familia boxplot / violín
resume cada distribución en una franja angosta. Son probablemente las dos visualizaciones
más rendidoras de toda la clase, y el violín es el boxplot con la **forma** de la
distribución dibujada encima.

Pero tienen un requisito. **Un violín es
una densidad, y una densidad es para variables continuas.** Por eso acá van sobre el
salario y no sobre el bienestar: entre dos salarios siempre existe uno intermedio, y entre
el 4 y el 5 de una escala de 1 a 7 no existe nada.

El tercer panel muestra lo que corresponde cuando la variable es ordinal: una barra por
cada valor que existe de verdad. Es la misma comparación entre grupos, sin inventar nada.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13.5, 3.8))
orden_educacion = ["Secundario", "Terciario", "Universitario", "Posgrado"]

# Barras horizontales: los nombres de los niveles no entran en el eje x.
sns.boxplot(data=emp, y="educacion_nivel", x="salario_millones",
            order=orden_educacion, color="#00529B", ax=axs[0])
axs[0].set_title("Boxplot: mediana, cuartiles, bigotes")
axs[0].set_xlabel("salario (millones de $)")
axs[0].set_ylabel("")

sns.violinplot(data=emp, y="educacion_nivel", x="salario_millones",
               order=orden_educacion, color="#00529B", ax=axs[1])
axs[1].set_title("Violin: ademas, la FORMA de la distribucion")
axs[1].set_xlabel("salario (millones de $)")
axs[1].set_ylabel("")

# La variable ordinal va con barras, una por cada valor que existe.
# discrete=True pone una barra por entero; multiple="dodge" las pone lado a lado;
# stat="percent" con common_norm=False saca el porcentaje DENTRO de cada grupo.
intervencion = bien[bien["fase"] == "intervencion"]
sns.histplot(data=intervencion, x="bienestar", hue="grupo_fruta", discrete=True,
             multiple="dodge", stat="percent", common_norm=False, ax=axs[2])
axs[2].set_title("Una escala de 1 a 7 va con barras")
axs[2].set_xlabel("bienestar (1 a 7)")
axs[2].set_xticks([1, 2, 3, 4, 5, 6, 7])

plt.tight_layout()
plt.savefig("figuras/clase03_box_violin.png", dpi=150, bbox_inches="tight")
plt.show()

El boxplot resume tan bien que **esconde cuánta gente hay detrás**. No
es razón para no usarlo (lo vamos a usar mucho), es razón para chequear los `n` antes de
mandarlo al informe, y si hay celdas chicas, decirlo o superponer los puntos.

### Un ejemplo para cerrar: la línea que afirma de más

El gráfico de líneas es cómodo, y muchas herramientas lo ofrecen por default incluso
cuando el mapeo no lo admite.

In [ ]:
# Contraejemplo: unir con una linea categorias que no tienen orden.
medias_area = bien.groupby("area", as_index=False)["bienestar"].mean()
sns.lineplot(data=medias_area, x="area", y="bienestar", marker="o")
plt.title("¿Que afirma esta linea?")
plt.show()

La línea **afirma continuidad**. Dice que entre Ingeniería y Ventas existen valores
intermedios, que uno podría pararse "en la mitad" entre las dos áreas y leer un bienestar.
Pero no existen, entre dos categorías no hay nada.

De ahí la regla: se unen puntos con una línea **sólo cuando el eje x es tiempo o alguna
otra cantidad continua**. Con categorías, puntos o barras.

### Relaciones x-y: un punto por persona

La línea de recién falló porque el eje x era una categoría. Cuando **las dos variables
son cantidades**, se abre otra rama del catálogo, la de *relaciones*, y su gráfico básico
es el **scatterplot** (o gráfico de dispersión): el que contesta la pregunta "¿estos dos
números van juntos?".

RRHH dejó una de esas preguntas dando vueltas: *¿el salario acompaña a la edad?* Cada
punto de la figura de abajo es un empleado, con su edad en el eje x y su sueldo en el eje y.

In [ ]:
# Dos cantidades, una en cada eje: un punto por empleado.
# alpha=0.5 los dibuja semitransparentes: son 600 y se pisan entre si.
sns.scatterplot(data=emp, x="edad", y="salario_millones", alpha=0.5)
plt.xlabel("edad (años)")
plt.ylabel("salario mensual (millones de $)")
plt.title("Edad y salario: un punto por empleado")
plt.show()

El scatterplot **no resume nada**. Muestra dos cosas a la vez, y conviene leerlas por separado.

La primera es la **forma** de la relación: el sueldo sube entre los veinte y los cuarenta,
después se aplana, y pasados los cincuenta parece aflojar un poco. No es una recta, y con
esta nube tampoco se puede afirmar mucho más que eso.

La segunda es la **dispersión**: entre los empleados de 45 años los
hay ganando 1,10 millones y los hay ganando 1,43. Esa distancia, *dentro* de una misma
edad parece bastante grande. Una barra
con el salario promedio por franja etaria habría contado lo primero y tapado lo segundo
por completo.

Y la regla de mapeo es la de la línea, dada vuelta: **scatterplot cuando las dos
variables son cantidades**. Si una de las dos es una categoría, la comparación vuelve a
la familia de barras, boxplots y violines.

### ✏️ Consigna 1: el gráfico que contesta cada pregunta

RRHH manda cuatro preguntas por mail. Completá la función de seaborn que contesta cada
una (el resto del código ya está armado):

1. *"¿Cuánta gente tiene cada área?"*
2. *"¿El bienestar se corrió entre fases en cada grupo?"*
3. *"¿Los que llevan más años ganan más?"*
4. *"¿Cómo se distribuye el salario?"*

In [ ]:
# TODO: completa la funcion de seaborn de cada pregunta
fig, axs = plt.subplots(2, 2, figsize=(11, 6.5))

orden_areas = empleados["area"].value_counts().index
sns.___(data=empleados, y="area", order=orden_areas, color="#00529B", ax=axs[0, 0])
axs[0, 0].set_title("1. gente por area")

sns.___(data=bien, x="fase", y="bienestar", hue="grupo_fruta", ax=axs[0, 1])
axs[0, 1].set_title("2. bienestar por fase y grupo")

sns.___(data=emp, x="antiguedad_anios", y="salario_millones", alpha=0.5, ax=axs[1, 0])
axs[1, 0].set_title("3. antiguedad vs salario")
axs[1, 0].set_ylabel("salario (millones de $)")

sns.___(data=emp, x="salario_millones", ax=axs[1, 1])
axs[1, 1].set_title("4. distribucion del salario")
axs[1, 1].set_xlabel("salario (millones de $)")

plt.tight_layout()
plt.show()

## 4. Las decisiones que cambian la historia

<a id="sec-decisiones"></a>

Hasta acá elegimos **qué** graficar. Ahora, las decisiones que cambian lo que el lector
concluye aunque el gráfico no cambie de tipo. Dos gráficos correctos del mismo dato
pueden contar historias distintas, y el informe obliga a elegir uno.

### La relación de aspecto

In [ ]:
serie = bien.groupby("semana", as_index=False)["bienestar"].mean()

# La MISMA serie, tres hojas de forma distinta. Solo cambia figsize (ancho, alto).
plt.figure(figsize=(10, 1.8))
sns.lineplot(data=serie, x="semana", y="bienestar", color="#00529B", marker="o")
plt.title("ancha y baja: 'apenas se movio'")
plt.savefig("figuras/clase03_aspecto_ancho.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(4, 4))
sns.lineplot(data=serie, x="semana", y="bienestar", color="#00529B", marker="o")
plt.title("cuadrada: neutral")
plt.savefig("figuras/clase03_aspecto_cuadrado.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(2, 4.5))
sns.lineplot(data=serie, x="semana", y="bienestar", color="#00529B", marker="o")
plt.title("angosta y alta, 'salto enorme'")
plt.savefig("figuras/clase03_aspecto_alto.png", dpi=150, bbox_inches="tight")
plt.show()
# Proba: cambia los figsize y volve a correr.

¿Cuál de las tres está mal? **Ninguna.** Son los mismos ocho números, con los mismos
ejes y las mismas etiquetas: las tres son visualizaciones válidas de los mismos datos. Y
sin embargo la primera sugiere que no pasó nada y la tercera sugiere un salto dramático.

Acá aparece el hilo de la clase en su forma más pura, pero cuidado con la interpretación más laxa del problema.
Wilke no dice que cada uno dibuje la historia que quiera. Dice que varias versiones son
igualmente válidas, y que la elección se justifica por **qué diferencias importa hacer
notar**. Es una libertad de elección, pero que debe hacerse a conciencia.

### El color: tres usos, tres familias de escala

> *"Hay tres usos fundamentales del color en visualización de datos: (i) podemos usar
> color para distinguir grupos de datos entre sí; (ii) podemos usar color para
> representar valores; y (iii) podemos usar color para destacar."* (Wilke, cap. 4)

Cada uso pide su familia de colores:

*   **cualitativa** para distinguir grupos sin orden,
*   **secuencial** para representar valores,
*   **de acento** para destacar.




Usar la familia
equivocada nos da un mal gráfico, no es un problema de estética.

Un mal uso por familia:

In [ ]:
# Distinguir MAL: una escala secuencial sobre una variable sin orden.
# reset_index convierte el conteo en una tabla con columnas "area" y "count".
conteo = empleados["area"].value_counts().reset_index()
sns.barplot(data=conteo, y="area", x="count", hue="area",
            palette="Blues", legend=False)
plt.title("¿Cual area 'es mas'? El azul oscuro insinua un orden que no existe")
plt.show()

Si alguien pensó RRHH, aunque sea un momento, es entendible: **una escala secuencial le inventó una
jerarquía a una variable que no la tiene**. Las áreas no van de menos a más; son cinco
categorías sin orden. Pero el azul oscuro se lee como "más" y el claro como "menos", así
que el gráfico está afirmando algo sobre los datos que los datos no dicen.

Es **malo**, no feo: el problema no es que combine mal, es que se percibe mal. Para
variables nominales va una paleta cualitativa, donde ningún color pesa más que otro.

In [ ]:
# Representar MAL: una escala divergente desbalanceada.
desvio = bien.pivot_table(index="sede", columns="semana", values="bienestar",
                          aggfunc="mean") - bien["bienestar"].mean()

fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.6))
sns.heatmap(desvio, cmap="RdBu_r", center=0, vmin=-0.45, vmax=0.45, ax=axs[0],
            cbar_kws={"label": "desvio"})
axs[0].set_title("Balanceada: mismo desvio, misma intensidad")
sns.heatmap(desvio, cmap="RdBu_r", vmin=-0.1, vmax=0.45, ax=axs[1],
            cbar_kws={"label": "desvio"})
axs[1].set_title("Desbalanceada: -0.1 grita, +0.1 susurra")
plt.tight_layout()
plt.show()

En el panel de la derecha, un desvío de −0,1 se ve intenso y uno de +0,1 casi no se ve.
Es el mismo número, con el mismo signo cambiado, y el ojo lee dos magnitudes distintas. Esta figura es
**incorrecta**.

Una escala divergente está compuesta por dos escalas secuenciales pegadas en un punto medio, y por eso tiene que
estar **balanceada**: la progresión hacia cada lado debe ser equivalente.

Ojo, lo que justifica su uso no es que el número tenga signo, sino que el punto medio tenga significado propio. Por ejemplo, acá el cero separa "por encima
del promedio" de "por debajo".



In [ ]:
# Destacar: gastar el color en UNA sola cosa. Atletas australianos (Telford &
# Cunningham 1991), el ejemplo del libro, con sus datos originales.
ais = pd.read_csv("https://vincentarelbundock.github.io/Rdatasets/csv/DAAG/ais.csv")
varones = ais[ais["sex"] == "m"]

# .isin: ¿el valor esta en esta lista? Devuelve True/False por fila (Clase 2).
es_de_pista = varones["sport"].isin(["T_400m", "T_Sprnt"])
pista = varones[es_de_pista]
otros = varones[~es_de_pista]    # la virgulilla ~ invierte True y False: "los que NO"

sns.scatterplot(data=otros, x="ht", y="pcBfat", color="#9FB0BD", alpha=0.7,
                label="otros deportes")
sns.scatterplot(data=pista, x="ht", y="pcBfat", color="#C0492F", s=70,
                label="atletismo (pista)")
plt.xlabel("altura (cm)")
plt.ylabel("% grasa corporal")
plt.title("Los atletas de pista, entre los mas bajos y magros")
plt.savefig("figuras/clase03_highlight_atletas.png", dpi=150, bbox_inches="tight")
plt.show()

¿Cuántos colores tiene esa figura? **Dos.** ¿Y cuántos grupos hay en los datos? Muchos
más.

La figura junta todos los otros deportes en un solo gris. Eso es destacar. Es **agotar el
color entero en una sola cosa**, y mandar todo lo demás al fondo de modo que no compita
por la atención.

La conclusión, que los de pista están entre los más bajos y magros, se lee más fácilmente.

In [ ]:
#@title ¿Qué familia de escala le corresponde a  cada una? {display-mode: "form"}
#@markdown **1. Sede**
sede = "elegir" #@param ["elegir", "cualitativa", "secuencial", "divergente"]
#@markdown **2. Años de antiguedad**
antiguedad_anios = "elegir" #@param ["elegir", "cualitativa", "secuencial", "divergente"]
#@markdown **3. Bienestar**
bienestar = "elegir" #@param ["elegir", "cualitativa", "secuencial", "divergente"]

respuestas = [("1. sede", sede, "cualitativa"),
              ("2. antiguedad_anios", antiguedad_anios, "secuencial"),
              ("3. bienestar", bienestar, "secuencial")]

for titulo, elegida, correcta in respuestas:
    if elegida == "elegir":
        print("⚪", titulo, "— todavía sin responder")
    elif elegida == correcta:
        print("✅", titulo, "—", elegida)
    else:
        print("❌", titulo, "—", elegida, "  ·  la respuesta es:", correcta)

### ✏️ Consigna 2: ¿cuál va al informe?

La consigna central de la clase. Abajo hay **tres versiones de la figura principal del
piloto**. Los números detrás de las tres son los mismos, pero una de las tres ya la
auditaste en la sección 1 y sabés cómo se llama: usala de filtro.

**Antes de decidir, mirá en qué número arranca el eje y de cada panel.**

Elegí cuál mandarías a RRHH y justificalo en dos líneas. Entre las que quedan no hay
una única respuesta correcta, pero hay justificaciones que no se sostienen.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12.5, 3.8))

# A: las distribuciones completas
sns.violinplot(data=bien, x="fase", y="bienestar", hue="grupo_fruta",
               cut=0, inner="quart", ax=axs[0], legend=False)
axs[0].set_title("A: distribuciones completas")
axs[0].set_ylabel("bienestar (1 a 7)")

# B: las medias con su intervalo de confianza del 95%
sns.pointplot(data=bien, x="fase", y="bienestar", hue="grupo_fruta",
              dodge=0.2, errorbar=("ci", 95), ax=axs[1])   # dodge separa los grupos
axs[1].set_title("B: medias e IC 95%")
axs[1].set_ylabel("bienestar (1 a 7)")

# C: las medias como barras
sns.barplot(data=bien, x="fase", y="bienestar", hue="grupo_fruta",
            errorbar=None, ax=axs[2], legend=False)
axs[2].set_ylim(4.4, 5.05)
axs[2].set_title("C: barras")
axs[2].set_ylabel("bienestar (1 a 7)")

plt.tight_layout()
plt.savefig("figuras/clase03_tres_versiones.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
eleccion = "__"
justificacion = "___"
print("Eleccion:", eleccion)
print(justificacion)

¿Ya elegiste? Corré la celda de abajo para ver los pros y los contras de las tres
opciones, y comparar tu justificación.

In [ ]:
#@title 🔎 Pros y contras de las tres opciones {display-mode: "form"}
# Los numeros salen de los datos, no estan escritos a mano.
_inter = bien.groupby(["fase", "grupo_fruta"])["bienestar"].mean()["intervencion"]
DIF_REAL = 100 * (_inter["Tratamiento"] / _inter["Control"] - 1)
PISO_C = 4.4                                              # el set_ylim del panel C
DISTORSION = (_inter["Tratamiento"] - PISO_C) / (_inter["Control"] - PISO_C)

OPCIONES = {
    "A": ("Violinplot",
          ["Muestra las distribuciones de los grupos.",
           "El eje va de 1 a 7, la escala real."],
          [f"El efecto real ({DIF_REAL:.1f}%) queda adentro del solape, y el lector "
           "puede concluir 'no paso nada'.",
           "No dice nada sobre las diferencias entre e intra fases."]),
    "B": ("Pointplot",
          ["Muestra el efecto Y su cambio de fase a fase.",
           "Un punto dice cuanto con su posición relativa, asi que acotar el eje es solo un zoom."],
          ["Resume 24.000 mediciones en cuatro puntos. La noción de grupo se pierde un poco",
           "El zoom hace que un tercio de punto ocupe media pantalla. Hay que aclarar el "
           "rango del eje en el epigrafe."]),
    "C": (f"Barras",
          ["Se lee de un vistazo, es la que mas se ve "
           "en presentaciones."],
          [f"Una barra dice cuanto con su longitud total: la de Tratamiento se ve "
           f"{DISTORSION:.1f} veces mas alta que la de Control, y la diferencia real "
           f"es del {DIF_REAL:.1f}%.",
           "Es incorrecta, no fea. Ya la auditaste en la seccion 1."]),
}

try:
    elegida = str(eleccion).strip().upper()
except NameError:
    elegida = ""
if elegida not in OPCIONES:
    elegida = None
    print("No elegiste A, B ni C. Completa `eleccion` arriba y volve a correr esta celda.\n")

for letra, (titulo, pros, contras) in OPCIONES.items():
    marca = "  <-- TU ELECCION" if letra == elegida else ""
    # \033[1m enciende la negrita y \033[0m la apaga.
    encabezado = f"{letra}: {titulo}{marca}"
    print(f"\033[1m{encabezado}\033[0m" if letra == elegida else encabezado)
    for p in pros:
        print("   + ", p)
    for c in contras:
        print("   - ", c)
    print()

Entre A y B no hay una unica respuesta correcta: son dos figuras honestas que
contestan preguntas distintas. Con C si hay una respuesta. Y ojo, lo que la hace
incorrecta a C NO es que el eje arranque arriba de cero, porque B
tambien arranca arriba de cero y esta bien. La diferencia es qué atributo
codifica el valor: la posicion admite zoom, la longitud no.

## 5. Componer y comunicar

<a id="sec-componer"></a>

El informe no va a tener una figura: va a tener varias. Dos reglas gobiernan cómo se
leen en conjunto.

### Regla 1: paneles que se comparan comparten escala

Para partir una figura en paneles (una por área, por sede, por grupo) seaborn tiene las
funciones *figure-level*: `relplot`, `catplot`, `displot`. Son la otra familia de
seaborn: las que veníamos usando (*axes-level*: `scatterplot`, `boxplot`, `histplot`...)
dibujan en **un** panel y por eso aceptan `ax=`; las figure-level fabrican la hoja
entera con todos sus paneles, y por eso `ax=` **no funciona** con ellas. Es la trampa
número uno de seaborn, tenela en cuenta cuando un error mencione `ax`.

El argumento `col=` crea un panel por categoría. Miralo dos veces, con un solo argumento
de diferencia:

In [ ]:
# Un panel por nivel educativo, y cada panel con SU PROPIO eje (sharey=False).
historico = empleados.merge(salario, on="empleado_id")
historico["salario_millones"] = historico["salario_mensual"] / 1_000_000

# El nivel educativo tiene un orden natural, y los paneles lo respetan.
orden_educacion = ["Secundario", "Terciario", "Universitario", "Posgrado"]

g = sns.relplot(data=historico, x="anio", y="salario_millones",
                col="educacion_nivel", col_order=orden_educacion,
                kind="line", estimator="mean", errorbar=None,
                height=2.6, aspect=0.95, facet_kws={"sharey": False})
g.set(xticks=[2023, 2024, 2025])
# set_titles: el titulo de cada panel, sin el "educacion_nivel =" adelante.
g.set_titles("{col_name}")
g.set_axis_labels("", "salario (millones de $)")
g.figure.suptitle("sharey=False: ¿que nivel educativo gana mas?", y=1.06)
plt.show()

Contestá antes de seguir: **¿qué nivel educativo gana más?**

Los cuatro paneles se ven casi iguales, y esa es la trampa: cada uno tiene **su propio
eje**. Con `sharey=False` seaborn estira cada panel hasta llenar su caja, así que los
cuatro suben lo mismo *en la pantalla* sin importar de dónde arrancan.

Wilke lo dice así:
la mente del lector espera que los ejes sean los mismos, y compara alturas entre paneles
sin verificar las escalas. Con ejes libres esa comparación es directamente inválida,
aunque cada panel por separado sea impecable.

In [ ]:
# Un solo argumento de diferencia: sin facet_kws, sharey vale True (el default).
g = sns.relplot(data=historico, x="anio", y="salario_millones",
                col="educacion_nivel", col_order=orden_educacion,
                kind="line", estimator="mean", errorbar=None,
                height=2.6, aspect=0.95)
g.set(xticks=[2023, 2024, 2025])
g.set_titles("{col_name}")
g.set_axis_labels("", "salario (millones de $)")
g.figure.suptitle("sharey=True (el default): ahora si se puede comparar", y=1.06)
plt.savefig("figuras/clase03_facetas.png", dpi=150, bbox_inches="tight")
plt.show()

Ahora sí, y la conclusión no es la que sugería la figura anterior. Los cuatro niveles
suben casi en paralelo, pero **arrancan en alturas muy distintas**: en 2025 un posgrado
promedia 1,36 millones y un secundario 1,13, un 20% de diferencia. Esa brecha, que es la
conclusión principal de la figura, era **invisible** con los ejes libres.

**¿Entonces `sharey=False` está prohibido?** No. A veces las magnitudes son tan distintas
que compartir eje aplasta todos los paneles menos uno. Pero en ese caso hay que
**avisarlo explícitamente en el epígrafe**, porque el lector no lo va a chequear solo.

Y otra regla del mismo capítulo, que esta figura ya aplica: los paneles van siempre en un
orden con sentido. Acá van de Secundario a Posgrado porque el nivel educativo tiene un
orden natural, y eso se pide con `col_order=`. Dejarlos en el orden alfabético con el que
vinieron habría sido, otra vez, una decisión no tomada.

A veces el hallazgo
**no está en ningún panel**, emerge al compararlos. Para ello, primero tienen que ser
efectivamente comparables.

### Regla 2: el mismo lenguaje visual en todo el informe

Si Tratamiento es terracota en la figura 1, es terracota en todas. Si las sedes van
ordenadas por tamaño en una figura, van así en todas. El test rápido: **contá las
leyendas**. Si necesitás dos leyendas para explicar la misma variable, el lenguaje
visual no es consistente.

### Regla 3: las palabras son parte del mapeo

> *"Los títulos y etiquetas de ejes y leyendas explican qué son los valores que se
> muestran y cómo se mapean a los atributos gráficos."* (Wilke, cap. 22)

Las etiquetas son la parte **verbal** del mapeo: dicen qué significa cada atributo. Se
pueden omitir solo cuando las etiquetas de los valores ya lo explican todo (una leyenda
"Femenino / Masculino" no necesita el título "género"; una "Control / Tratamiento" sin
título es un enigma).

El principio
de la cátedra es más exigente que el del libro: **el título de la figura es la conclusión**, porque el
que decide lee el título antes que los ejes.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 3.8), sharey=True)
piloto = bien.groupby(["semana", "grupo_fruta"], as_index=False)["bienestar"].mean()

sns.lineplot(data=piloto, x="semana", y="bienestar", hue="grupo_fruta",
             marker="o", ax=axs[0], legend=False)
axs[0].set_xlabel(""); axs[0].set_ylabel("")
axs[0].set_title("Sin palabras: el lector adivina")

sns.lineplot(data=piloto, x="semana", y="bienestar", hue="grupo_fruta",
             marker="o", ax=axs[1])
axs[1].set_xlabel("semana del piloto"); axs[1].set_ylabel("bienestar promedio (1-7)")
axs[1].legend(title="grupo")
axs[1].set_title("El bienestar sube solo en el grupo con fruta")

plt.tight_layout()
plt.savefig("figuras/clase03_titulos.png", dpi=150, bbox_inches="tight")
plt.show()
# "It is a bad practice to make your readers guess what you mean" (Wilke).

## 6. El informe

<a id="sec-informe"></a>

Última pieza: convertir el análisis en **una** figura que encabece el informe. El
capítulo 29 de Wilke pide contar una historia con estructura de tensión y resolución, y
deja dos frases que funcionan como resumen de toda la clase:

> *"Cuando intentás mostrar demasiados datos a la vez, podés terminar no mostrando
> nada."* (Wilke, cap. 29)

> *"Simple y claro le gana a complejo y confuso."* (Wilke, cap. 29)

Primero la versión que intenta mostrarlo todo. Tomate veinte segundos mirándola antes de seguir. ¿Qué conclusión te llevas?


In [ ]:
sns.scatterplot(data=emp, x="antiguedad_anios", y="salario_millones",
                hue="area", size="edad", style="genero", alpha=0.6)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.xlabel("antigüedad (años)")
plt.ylabel("salario mensual (millones de $)")
plt.title("Cinco variables en un grafico. ¿Que conclusion te llevas?")
plt.show()

La respuesta honesta es **ninguna**. Hay cinco variables mapeadas ahí adentro (posición
en x, posición en y, color, tamaño y forma), tres leyendas, y cero historia.

No es necesario graficar todas las dimensiones que uno tiene. Hay que graficar **la que
responde la pregunta**. Y la pregunta de RRHH ya la tenemos.

In [ ]:
# La figura del informe: la pregunta de RRHH, contestada y anotada.
piloto = bien.groupby(["semana", "grupo_fruta"], as_index=False)["bienestar"].mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.lineplot(data=piloto, x="semana", y="bienestar", hue="grupo_fruta",
             marker="o", ax=ax)
ax.axvline(4.5, color="#9FB0BD", linestyle="--")            # linea vertical
ax.text(4.65, 4.53, "empieza la fruta", color="#57708a", fontsize=9)   # texto encima
ax.set_xlabel("semana del piloto")
ax.set_ylabel("bienestar promedio (escala 1-7)")
ax.legend(title="grupo")
ax.set_title("El bienestar sube solo en el grupo con fruta: +0,34 puntos, un efecto chico y consistente")

plt.savefig("figuras/clase03_figura_informe.png", dpi=150, bbox_inches="tight")
plt.show()

### La última decisión: el título

Tres títulos posibles para esa figura. Ninguno contradice los números:

| título | problema |
|---|---|
| "Bienestar por fase y grupo" | describe el contenido, no dice nada |
| "El programa de fruta mejora el bienestar de los empleados" | dice más de lo que el dato sostiene |
| "El bienestar sube solo en el grupo con fruta: un efecto chico y consistente" | **el más sobrio** |

El segundo es el tentador: es el título que un equipo de datos manda cuando quiere que
su proyecto siga vivo. Sostener el matiz (*subió, es consistente, es chico*) es más
difícil que exagerarlo, pero es exactamente el trabajo que hay que hacer.

### Y esto era, además, un reporte reproducible

Una última cosa, recordemos que **esta notebook
es el informe**. Tiene el texto, el código que produjo cada número, las figuras y la
conclusión. Cualquiera puede volver a correrla de arriba a abajo y obtener lo mismo, y
si mañana llegan datos nuevos, se corre de nuevo y las figuras se regeneran. Eso es un
reporte reproducible: no un documento *sobre* el análisis, sino el análisis mismo.

Para mandarlo: en Colab, `Archivo → Descargar → .ipynb` (el código vivo) o
`Archivo → Imprimir → PDF` (la foto para quien no corre código).

### ✏️ Consigna bonus track

RRHH quedó conforme y manda otro pedido, esta vez sobre **rotación de personal**. El
dataset es `hr_attrition` (IBM): 1.470 empleados, 35 columnas, y una columna `Attrition` que dice si la persona dejó la
empresa.

**El pedido**: una (1) figura que sostenga una afirmación sobre quiénes se van, con
título-conclusión, ejes etiquetados y guardada a archivo. Sugerencia de arranque:
¿los que se van ganan distinto (`MonthlyIncome`) que los que se quedan?

**Esta consigna no la vamos a hacer en clase**: es el único tramo largo y no entra en
los tiempos. Queda como práctica por tu cuenta, con el enunciado y los datos ya listos
acá abajo. Es el mejor repaso de todo lo de hoy.

In [ ]:
URL_HR = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv"
hr = pd.read_csv(URL_HR)
print(hr.shape)
hr[["Age", "Attrition", "Department", "MonthlyIncome", "JobRole", "OverTime"]].head()

In [ ]:
medianas = hr.groupby("Attrition")["MonthlyIncome"].median()

# TODO: elegi el grafico Y el mapeo que sostienen tu afirmacion
# (pista: horizontal se lee mejor cuando las etiquetas son texto, como en la seccion 1)
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.___(data=hr, y="___", x="___", ax=ax)
ax.set_xlabel("___")
ax.set_ylabel("___")
# TODO: el titulo es la conclusion, no la descripcion
ax.set_title("___")

plt.savefig("figuras/clase03_cierre_attrition.png", dpi=150, bbox_inches="tight")
plt.show()
print(medianas.round(0))

## Resumen de hoy

| Paso | Qué usamos | En una línea |
|---|---|---|
| **El criterio** | feo / malo / incorrecto | estética, percepción o matemática: tres fallas distintas, tres gravedades |
| **El mapeo** | `x=`, `y=`, `hue=`, `style=`, `size=` | un gráfico es un mapeo de datos a atributos; la escala debe ser uno a uno |
| **El repertorio** | `countplot`, `histplot`, `kdeplot`, `boxplot`, `violinplot`, `scatterplot` | primero la pregunta, después el gráfico |
|  | `binwidth=`, `cut=0`, `order=` | los defaults deciden por vos: el bin, la cola imposible, el orden |
| **Las decisiones** | paletas cualitativa / secuencial / divergente | cada uso del color pide su familia; el highlight gasta el color en una sola cosa |
| **Componer** | `relplot(col=...)`, `sharey` | paneles que se comparan comparten escala; mismo lenguaje visual en todo el informe |
| **Comunicar** | `set_title`, `set_xlabel`, `savefig` | las palabras son parte del mapeo; el título es la conclusión |

Mini apéndice para cuando lo necesites:la [galería de seaborn](https://seaborn.pydata.org/examples/index.html) +
[from Data to Viz](https://www.data-to-viz.com/) como repertorio de consulta.

**Lo que nos llevamos hoy:** dos gráficos correctos del mismo dato pueden contar
historias distintas. Elegir cuál mandar es una decisión de comunicación con
consecuencias, y esa decisión, como las de la Clase 2, debe ser a conciencia.